# 02 - Data Cleaning
**Goal:** turn the raw CSV into an analysis-ready file saved at `data/processed/netflix_cleaned.csv`.

In [3]:
import pandas as pd

df = pd.read_csv("C:\\Users\\sreej\\codetech_projects\\NetflixContentVisualization\\netflix_content_visualization\\data\\raw\\netflix_titles.csv")
print("Raw shape:", df.shape)

Raw shape: (8807, 12)


## 1. Remove duplicates

In [4]:
df = df.drop_duplicates().copy()
print("After dropping duplicates:", df.shape)

After dropping duplicates: (8807, 12)


## 2. Handle missing values

In [13]:
# Text columns: label missing values instead of dropping thousands of rows
for col in ["director", "cast", "country"]:
    df[col] = df[col].fillna("Unknown")

# A few rows have the duration typed into the `rating` column - move it back
mask = df["rating"].str.contains("min", na=False)
df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = "Unknown"
df["rating"] = df["rating"].fillna("Unknown")

df.isnull().sum()

show_id               0
type                  0
title                 0
director              0
cast                  0
country               0
date_added            0
release_year          0
rating                0
duration              0
listed_in             0
description           0
year_added            0
month_added           0
month_name            0
duration_value        0
duration_min       2666
seasons               0
primary_country       0
primary_genre         0
dtype: int64

## 3. Fix dates

In [6]:
# Some values have leading spaces, so strip before parsing
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), errors="coerce")

# Only a handful of rows lack a date - safe to drop for time-based analysis
df = df.dropna(subset=["date_added"]).copy()

df["year_added"] = df["date_added"].dt.year.astype(int)
df["month_added"] = df["date_added"].dt.month.astype(int)
df["month_name"] = df["date_added"].dt.month_name()

df[["date_added", "year_added", "month_added", "month_name"]].head()

,date_added,year_added,month_added,month_name
0,2021-09-25,2021,9,September
1,2021-09-24,2021,9,September
2,2021-09-24,2021,9,September
3,2021-09-24,2021,9,September
4,2021-09-24,2021,9,September


## 4. Split `duration` into numbers

In [7]:
# "90 min" -> 90 (movies) ; "2 Seasons" -> 2 (TV shows)
df["duration_value"] = pd.to_numeric(df["duration"].str.extract(r"(\d+)")[0])

df["duration_min"] = df["duration_value"].where(df["type"] == "Movie")
df["seasons"] = df["duration_value"].where(df["type"] == "TV Show")

df[["type", "duration", "duration_min", "seasons"]].sample(5, random_state=1)

,type,duration,duration_min,seasons
7292,Movie,78 min,78.0,NaN
5855,TV Show,2 Seasons,NaN,2.0
7030,Movie,85 min,85.0,NaN
3458,TV Show,1 Season,NaN,1.0
7939,Movie,133 min,133.0,NaN


## 5. Helper columns for multi-value fields

In [8]:
# `country` and `listed_in` can hold several comma-separated values.
# Keep only the first one as a simple "main" value; we'll explode the full lists in the visualization notebook.
df["primary_country"] = df["country"].str.split(",").str[0].str.strip()
df["primary_genre"] = df["listed_in"].str.split(",").str[0].str.strip()

## 6. Sanity checks and save

In [9]:
print("Final shape:", df.shape)
print(df.isnull().sum()[lambda s: s > 0])   # nulls left (expected: duration_min / seasons)
df["type"].value_counts()

Final shape: (8797, 20)
duration_min    2666
seasons         6131
dtype: int64


type
Movie      6131
TV Show    2666
Name: count, dtype: int64

In [10]:
df.to_csv("C:\\Users\\sreej\\codetech_projects\\NetflixContentVisualization\\netflix_content_visualization\\data\\processed\\netflix_cleaned.csv", index=False)
print("Saved cleaned data.")

Saved cleaned data.
